# Notebook 61 — Telemetry Dataset

**Telemetry datasets specify learned controller inputs.**

Notebook 61 opens the third systems arc. The second arc built controller → infrastructure → distributed scheduling → cluster optimization. This notebook converts that optimized serving loop into a dataset for learning controllers safely.

In [ ]:
from pathlib import Path
import json, math, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from matplotlib.colors import ListedColormap, BoundaryNorm

OUT = Path("figures")
OUT.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (14, 8),
    "font.size": 18,
    "axes.titlesize": 32,
    "axes.labelsize": 24,
    "legend.fontsize": 16,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
})

## Helper functions

The notebook uses simple diagram primitives so every figure is reproducible from code.

In [ ]:
def rounded_box(ax, xy, w, h, text, fs=18, lw=2.2):
    x, y = xy
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=lw, edgecolor="black", facecolor="white"
    )
    ax.add_patch(patch)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fs)
    return patch

def arrow(ax, start, end, lw=2.0, rad=0.0):
    arr = FancyArrowPatch(
        start, end, arrowstyle="->", mutation_scale=18,
        linewidth=lw, color="black", connectionstyle=f"arc3,rad={rad}"
    )
    ax.add_patch(arr)
    return arr

def savefig(name):
    path = OUT / name
    plt.savefig(path, bbox_inches="tight", dpi=180)
    plt.show()
    return path

def horizontal_chain(ax, labels, y=0.5, x0=0.04, x1=0.96, box_h=0.18, fs=17):
    n = len(labels)
    gap = 0.012
    w = (x1 - x0 - gap*(n-1)) / n
    boxes = []
    for i, label in enumerate(labels):
        x = x0 + i*(w+gap)
        boxes.append(rounded_box(ax, (x, y), w, box_h, label, fs=fs))
        if i < n-1:
            arrow(ax, (x+w, y+box_h/2), (x+w+gap*0.92, y+box_h/2), lw=2.0)
    return boxes

## 1. Repository transition

Notebook 61 bridges cluster optimization into learning controllers.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.set_title("Notebook 61 bridges optimization and learning", pad=18)

second = ["43\nResource\nAllocation", "49\nAdaptive\nInfrastructure", "53\nDistributed\nScheduling", "59\nCluster\nOptimization"]
horizontal_chain(ax, second, y=0.66, x0=0.10, x1=0.90, box_h=0.16, fs=15)
ax.text(0.5, 0.58, "second arc: controller → infrastructure → distributed scheduling → cluster optimization",
        ha="center", va="center", fontsize=18)
ax.plot([0.08, 0.92], [0.51, 0.51], color="black", lw=1.8)

third = ["61\nTelemetry\nDataset", "67\nPolicy\nLearning", "71\nOffline\nEvaluation", "73\nSafety\nBounds", "79\nAdaptive\nController"]
horizontal_chain(ax, third, y=0.27, x0=0.05, x1=0.95, box_h=0.17, fs=15)
ax.text(0.5, 0.18, "third arc: telemetry dataset → policy learning → evaluation → safety → controller",
        ha="center", va="center", fontsize=18)

arrow(ax, (0.70, 0.66), (0.20, 0.44), lw=2.0, rad=-0.2)
ax.text(0.38, 0.47, "optimized serving traces become training data", ha="center", fontsize=17)

savefig("61_repository_transition_v2.png")

## 2. Third arc roadmap

The third arc uses telemetry to learn policies before allowing adaptive runtime control.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.set_title("Telemetry dataset opens the learning-controller arc", pad=18)

labels = ["61\nTelemetry\nDataset", "67\nPolicy\nLearning", "71\nOffline\nEvaluation", "73\nSafety\nBounds", "79\nAdaptive\nController"]
horizontal_chain(ax, labels, y=0.50, x0=0.08, x1=0.92, box_h=0.19, fs=16)

rounded_box(ax, (0.18, 0.18), 0.64, 0.13,
            "Third arc: use telemetry to learn policies before allowing adaptive runtime control.",
            fs=17)
savefig("61_third_arc_roadmap_v2.png")

## 3. Event streams become aligned samples

Runtime logs are event streams. Controller learning needs time-indexed rows.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.set_title("Event streams become time-indexed tables", pad=18)

events = ["queue update", "GPU sample", "memory sample", "policy decision", "latency result", "constraint flag"]
for i, ev in enumerate(events):
    y = 0.73 - i*0.085
    rounded_box(ax, (0.08, y), 0.28, 0.065, ev, fs=16)
    arrow(ax, (0.36, y+0.033), (0.49, 0.50), lw=1.5)

arrow(ax, (0.49, 0.50), (0.55, 0.50), lw=2.2)

# table
x0, y0, w, h = 0.58, 0.32, 0.34, 0.34
cols = ["t", "S(t)", "π(t)", "S(t+1)", "y(t)", "c(t)"]
rows = 5
cw = w/len(cols)
rh = h/(rows+1)
for j, col in enumerate(cols):
    ax.add_patch(Rectangle((x0+j*cw, y0+h-rh), cw, rh, fill=False, lw=1.5))
    ax.text(x0+j*cw+cw/2, y0+h-rh/2, col, ha="center", va="center", fontsize=15)
for r in range(rows):
    for j in range(len(cols)):
        ax.add_patch(Rectangle((x0+j*cw, y0+r*rh), cw, rh, fill=False, lw=1.2))
        ax.text(x0+j*cw+cw/2, y0+r*rh+rh/2, "·", ha="center", va="center", fontsize=18)

ax.text(0.5, 0.16, "Raw logs become aligned samples for controller training.",
        ha="center", fontsize=20)
savefig("61_event_streams_to_tables_v2.png")

## 4. State vector construction

Telemetry fields are mapped into controller inputs.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.set_title("State vector maps live telemetry into controller inputs", pad=18)

features = [
    ("q(t)\nqueue depth", 0.18, 0.70),
    ("u(t)\nutilization", 0.50, 0.78),
    ("m(t)\nmemory", 0.82, 0.70),
    ("λ(t)\narrival rate", 0.18, 0.38),
    ("r(t)\nreserve", 0.50, 0.30),
    ("v(t)\nverification", 0.82, 0.38),
]
for label, x, y in features:
    rounded_box(ax, (x-0.11, y-0.045), 0.22, 0.09, label, fs=15)
    arrow(ax, (x, y-0.05), (0.50, 0.53), lw=1.6)

rounded_box(ax, (0.38, 0.47), 0.24, 0.13, "State vector\nS(t)", fs=20, lw=2.5)
ax.text(0.5, 0.16, r"$S(t) = [q(t), u(t), m(t), \lambda(t), r(t), v(t)]$",
        ha="center", fontsize=26)
arrow(ax, (0.50, 0.47), (0.50, 0.28), lw=2.0)
rounded_box(ax, (0.36, 0.05), 0.28, 0.10, "policy input\nπ(S)", fs=18)

savefig("61_state_vector_v2.png")

## 5. Outcome labels

The label combines gain and constraint costs.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.set_title("Outcome labels combine gain and constraint costs", pad=18)

parts = [
    ("throughput\nchange ΔG", 0.14),
    ("latency\nchange ΔL", 0.32),
    ("memory\nchange ΔM", 0.50),
    ("spillover\nchange ΔS", 0.68),
    ("violation\nchange ΔV", 0.86),
]
for label, x in parts:
    rounded_box(ax, (x-0.095, 0.57), 0.19, 0.11, label, fs=15)
    arrow(ax, (x, 0.57), (0.50, 0.41), lw=1.8)

rounded_box(ax, (0.32, 0.30), 0.36, 0.12, "outcome label\ny(t)", fs=22)
ax.text(0.5, 0.12, r"$y(t)=\Delta G(t)-\Delta L(t)-\Delta M(t)-\Delta S(t)-\Delta V(t)$",
        ha="center", fontsize=26)
savefig("61_outcome_labels_v2.png")

## 6. Windowed telemetry

A learned controller often needs temporal context, not only a single instantaneous state.

In [ ]:
np.random.seed(61)
t = np.arange(90)
queue = 32 + 9*np.sin(t/14) + np.random.normal(0, 3.2, size=len(t))
queue = np.clip(queue, 12, 48)

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_title("Windowed telemetry gives the controller temporal context", pad=18)
ax.plot(t, queue, marker="o", linewidth=2, label="queue depth")
windows = [(8,24), (26,42), (44,60), (62,78)]
for a,b in windows:
    ax.axvspan(a, b, alpha=0.12)
    ax.text((a+b)/2, max(queue)+1.5, f"window\n{a}:{b}", ha="center", va="bottom", fontsize=13)
ax.set_xlabel("time step")
ax.set_ylabel("queue depth")
ax.grid(True, alpha=0.3)
ax.legend()
savefig("61_windowed_telemetry_v2.png")

## 7. Time-aware train/validation/test split

Telemetry must be split by time so the controller does not learn from future observations.

In [ ]:
np.random.seed(7)
t = np.arange(240)
q = 31 + 8*np.sin(t/19) + 4*np.sin(t/47) + np.random.normal(0, 2.7, size=len(t))
q = np.clip(q, 12, 48)

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_title("Time-aware split avoids leakage", pad=18)
ax.plot(t, q, linewidth=2, label="queue depth")
splits = [(0,155,"train"), (155,195,"validation"), (195,240,"test")]
for a,b,name in splits:
    ax.axvspan(a,b, alpha=0.13, label=name)
for x in [155,195]:
    ax.axvline(x, color="black", linestyle="--", linewidth=1.8)
ax.set_xlabel("time index")
ax.set_ylabel("queue depth")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right")
savefig("61_time_aware_split_v2.png")

## 8. Dataset quality pipeline

Quality checks protect learned-controller inputs.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.set_title("Quality checks protect learned-controller inputs", pad=18)

stages = [
    ("Raw telemetry", 0.40, 0.76, 0.20, 0.10),
    ("Missing values\nDuplicate events\nClock alignment", 0.29, 0.55, 0.42, 0.13),
    ("Constraint coverage\nPolicy balance\nOutlier removal", 0.29, 0.34, 0.42, 0.13),
    ("Validated dataset D", 0.33, 0.13, 0.34, 0.11),
]
for text, x, y, w, h in stages:
    rounded_box(ax, (x, y), w, h, text, fs=18)
for i in range(len(stages)-1):
    x, y, w, h = stages[i][1], stages[i][2], stages[i][3], stages[i][4]
    x2, y2, w2, h2 = stages[i+1][1], stages[i+1][2], stages[i+1][3], stages[i+1][4]
    arrow(ax, (x+w/2, y), (x2+w2/2, y2+h2), lw=2.2)

savefig("61_quality_pipeline_v2.png")

## 9. Operating-regime coverage

Dataset coverage must balance operating regimes. Rare but important regimes require targeted collection.

In [ ]:
regimes = ["steady", "rebalance", "reserve", "scale-out", "shed/shorten"]
counts = [148, 90, 12, 7, 1]

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_title("Dataset coverage must balance operating regimes", pad=18)
bars = ax.bar(regimes, counts)
ax.set_xlabel("operating regime")
ax.set_ylabel("samples")
ax.grid(axis="y", alpha=0.3)
ax.tick_params(axis="x", rotation=25)

for i, c in enumerate(counts):
    ax.text(i, c + 3, str(c), ha="center", fontsize=16)

ax.annotate("rare regimes\nneed targeted collection",
            xy=(4, 1), xytext=(3.0, 55),
            arrowprops=dict(arrowstyle="->", lw=2),
            fontsize=18, ha="center")
savefig("61_regime_coverage_v2.png")

## 10. Policy outcome surface

Samples should cover high-value regions and constraint-risk regions.

In [ ]:
np.random.seed(9)
x = np.linspace(0.05, 0.95, 160)
y = np.linspace(0.05, 0.95, 160)
X, Y = np.meshgrid(x, y)

Z = 0.55 + 0.42*np.exp(-((X-0.62)**2/0.06 + (Y-0.62)**2/0.08))
Z -= 0.75*((X < 0.20) & (Y > 0.82))
Z -= 0.26*((X > 0.75) & (Y < 0.30))
Z -= 0.16*(X < 0.18)

samples_x = np.linspace(0.10, 0.52, 34) + np.random.normal(0, 0.018, 34)
samples_y = 0.98 - samples_x + np.random.normal(0, 0.035, 34)
samples_y = np.clip(samples_y, 0.08, 0.94)

fig, ax = plt.subplots(figsize=(12, 9))
ax.set_title("Policy outcome surface", pad=18)
cf = ax.contourf(X, Y, Z, levels=12)
ax.scatter(samples_x, samples_y, s=55, edgecolor="black", zorder=3)
ax.add_patch(Rectangle((0.05, 0.82), 0.15, 0.13, fill=False, lw=3, linestyle="--"))
ax.text(0.13, 0.88, "constraint\nrisk", fontsize=18)
ax.text(0.58, 0.63, "high-value\ntraining region", fontsize=19)
ax.set_xlabel("reserve capacity")
ax.set_ylabel("active load")
cbar = fig.colorbar(cf, ax=ax)
cbar.set_label("expected outcome label")
savefig("61_policy_outcome_surface_v2.png")

## 11. Learning dataset specifies policy optimization

Notebook 61 closes by handing a validated dataset to Notebook 67.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.set_title("Learning dataset specifies policy optimization", pad=18)

labels = ["Runtime\ncluster", "Telemetry", "Dataset D", "Policy\nlearning", "Offline\nevaluation", "Verified\ncontroller", "Deployment\ngate"]
horizontal_chain(ax, labels, y=0.52, x0=0.04, x1=0.96, box_h=0.18, fs=15)

rounded_box(ax, (0.20, 0.18), 0.60, 0.12,
            "Dataset engineering is the bridge from optimized serving to learned runtime control.",
            fs=17)
savefig("61_learning_dataset_to_policy_v2.png")

## 12. Final synthesis

Telemetry datasets specify learned controller inputs.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.set_title("Telemetry datasets specify learned controller inputs", pad=18)

labels = ["cluster\ntelemetry", "trace\nschema", "state\nwindows", "action +\noutcome labels",
          "training\ndataset", "policy\nlearning", "validated\ncontroller"]
horizontal_chain(ax, labels, y=0.50, x0=0.02, x1=0.98, box_h=0.20, fs=16)

rounded_box(ax, (0.18, 0.17), 0.64, 0.12,
            "Notebook 61 converts runtime traces into state, action, outcome, and constraint records.",
            fs=17)
savefig("61_final_synthesis_v2.png")

## Download artifacts

The cell below packages this notebook's figures and confirms the output files for download.

In [ ]:
zip_path = Path("notebook_61_figures_v2.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.glob("61_*_v2.png")):
        zf.write(p, arcname=p.name)

manifest = {
    "notebook": "61_telemetry_dataset_v2",
    "figures": [p.name for p in sorted(OUT.glob("61_*_v2.png"))],
    "zip": str(zip_path),
}
Path("notebook_61_manifest_v2.json").write_text(json.dumps(manifest, indent=2))
manifest